# Bonus 02 — LangGraph durable workflows

A graph becomes valuable when the work must survive pauses, expose its state, and support a controlled correction. This lab builds a deterministic refund workflow so the runtime is the lesson. There is no LLM call and no API cost.

By the end you will be able to explain and use:

- parallel graph branches and reducers;
- super-steps, checkpoints, threads, and state history;
- dynamic human approval with `interrupt` and `Command`;
- safe resume behavior and the idempotency rule;
- time travel by forking from an earlier checkpoint.


## 1. Learn — the runtime is the product

Module 09 drew a model-and-tools loop and paused once for approval. Here the business process is ordinary software. LangGraph contributes execution state, routing, checkpoints, inspection, and resume.

```mermaid
flowchart LR
    S([START]) --> N[normalize]
    N --> P[policy check]
    N --> A[amount check]
    P --> D[decide]
    A --> D
    D -->|no signals| AA[auto approve]
    D -->|signals| H{{human review interrupt}}
    AA --> F[finalize]
    H --> F
    F --> E([END])
```

The policy and amount checks run in the same **super-step**. Both return small state updates. Reducers tell LangGraph how to merge their list updates without one branch erasing the other.


### Seven words that make the trace readable

| Word | Plain-language meaning |
|---|---|
| State | The typed record carried through the workflow |
| Update | The keys one node returns; usually not the entire state |
| Reducer | The merge rule for repeated or concurrent updates |
| Super-step | One graph tick; independent nodes in it may run in parallel |
| Checkpoint | A saved state snapshot at a super-step boundary |
| Thread | The checkpoint timeline selected by a `thread_id` |
| Fork | A new checkpoint branch created from an earlier snapshot |

A thread is not a Python thread. It is the durable identity of one workflow execution. A checkpoint is not a log line. It contains the state and what should run next.


## 2. Do — build the workflow

The course environment already includes LangGraph. `InMemorySaver` is appropriate for learning and tests; it disappears with this Python process. A production service uses a durable checkpointer such as Postgres.


In [ ]:
import operator
from importlib.metadata import version
from typing import Annotated, Literal, TypedDict

from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.types import Command, interrupt

print('langgraph:', version('langgraph'))
print('checkpointer:', InMemorySaver.__name__)


### State and reducers

Most fields use the default rule: a new value replaces the old one. `signals` and `audit_log` use `operator.add`, so new lists append. That matters because two checks will update those fields in the same super-step.

Reducers are policy. An append-only reducer is sensible for an audit trail, but wrong for a field such as `amount_usd` that must be corrected by replacement.


In [ ]:
class RefundState(TypedDict):
    case_id: str
    amount_usd: float
    days_since_purchase: int
    item_opened: bool
    signals: Annotated[list[str], operator.add]
    audit_log: Annotated[list[str], operator.add]
    needs_review: bool
    decision: str


def new_case(case_id: str, amount_usd: float, days: int, opened: bool) -> RefundState:
    return {
        'case_id': case_id,
        'amount_usd': amount_usd,
        'days_since_purchase': days,
        'item_opened': opened,
        'signals': [],
        'audit_log': [],
        'needs_review': False,
        'decision': '',
    }


### Nodes return updates

Each node is a small function. The two checks do not mutate the state and do not return unrelated keys. The human-review node interrupts **before** any external side effect.

That placement is important: when a run resumes, LangGraph restarts the interrupted node from its beginning. Code before `interrupt(...)` can therefore run more than once and must be idempotent.


In [ ]:
def normalize(state: RefundState) -> dict:
    return {
        'amount_usd': round(float(state['amount_usd']), 2),
        'audit_log': ['normalized'],
    }


def policy_check(state: RefundState) -> dict:
    signals = []
    if state['days_since_purchase'] > 30:
        signals.append('outside_30_day_window')
    if state['item_opened']:
        signals.append('opened_item')
    return {'signals': signals, 'audit_log': ['policy_check']}


def amount_check(state: RefundState) -> dict:
    signals = ['high_amount'] if state['amount_usd'] >= 500 else []
    return {'signals': signals, 'audit_log': ['amount_check']}


def decide(state: RefundState) -> dict:
    needs_review = bool(state['signals'])
    route = 'human' if needs_review else 'auto'
    return {'needs_review': needs_review, 'audit_log': [f'routed:{route}']}


def choose_route(state: RefundState) -> Literal['auto_approve', 'human_review']:
    return 'human_review' if state['needs_review'] else 'auto_approve'


def auto_approve(state: RefundState) -> dict:
    return {'decision': 'approved', 'audit_log': ['auto:approved']}


def human_review(state: RefundState) -> dict:
    answer = interrupt(
        {
            'case_id': state['case_id'],
            'amount_usd': state['amount_usd'],
            'signals': state['signals'],
            'question': 'approve or reject?',
        }
    )
    if answer not in {'approve', 'reject'}:
        raise ValueError('answer must be approve or reject')
    decision = 'approved' if answer == 'approve' else 'rejected'
    return {'decision': decision, 'audit_log': [f'human:{answer}']}


def finalize(state: RefundState) -> dict:
    return {'audit_log': [f"final:{state['decision']}"]}


### Wire, compile, and inspect

The list passed to `add_edge` creates a fan-in barrier: `decide` waits until **both** checks finish. Compiling with a checkpointer enables checkpoint history, interrupts, resume, and time travel.


In [ ]:
builder = StateGraph(RefundState)
builder.add_node('normalize', normalize)
builder.add_node('policy_check', policy_check)
builder.add_node('amount_check', amount_check)
builder.add_node('decide', decide)
builder.add_node('auto_approve', auto_approve)
builder.add_node('human_review', human_review)
builder.add_node('finalize', finalize)

builder.add_edge(START, 'normalize')
builder.add_edge('normalize', 'policy_check')
builder.add_edge('normalize', 'amount_check')
builder.add_edge(['policy_check', 'amount_check'], 'decide')
builder.add_conditional_edges('decide', choose_route)
builder.add_edge('auto_approve', 'finalize')
builder.add_edge('human_review', 'finalize')
builder.add_edge('finalize', END)

checkpointer = InMemorySaver()
refund_graph = builder.compile(checkpointer=checkpointer)
print(refund_graph.get_graph().draw_mermaid())


### Run a low-risk case and stream updates

`stream_mode='updates'` emits only what each node returned. It is often easier to debug than repeatedly printing the full state. The two check events may arrive in either order because they share a super-step. The reducer preserves both.


In [ ]:
auto_config = {'configurable': {'thread_id': 'bonus-case-101'}}
auto_case = new_case('case-101', amount_usd=45, days=10, opened=False)

for update in refund_graph.stream(
    auto_case,
    auto_config,
    stream_mode='updates',
):
    print(update)

auto_result = refund_graph.get_state(auto_config).values
print('decision:', auto_result['decision'])
print('signals:', auto_result['signals'])
print('audit:', auto_result['audit_log'])


## 3. Observe — checkpoints are executable state

`get_state_history` returns newest first. Each snapshot includes the values, metadata, and `next`: the nodes scheduled after that checkpoint. This is why a checkpoint can support resume or replay rather than acting as a plain audit message.


In [ ]:
auto_history = list(refund_graph.get_state_history(auto_config))
print('checkpoints:', len(auto_history), '(newest first)')
for snapshot in auto_history:
    print(
        'step=', snapshot.metadata.get('step'),
        'next=', snapshot.next,
        'decision=', snapshot.values.get('decision'),
    )


### Pause a high-value case

The same graph takes another edge when a check emits a signal. `interrupt` returns a JSON-serializable review packet to the caller. It does not block the notebook waiting for keyboard input, and it does not approve anything.


In [ ]:
review_config = {'configurable': {'thread_id': 'bonus-case-102'}}
review_case = new_case('case-102', amount_usd=1250, days=10, opened=False)
paused = refund_graph.invoke(review_case, review_config)

review_packet = paused['__interrupt__'][0].value
print('review packet:', review_packet)

paused_snapshot = refund_graph.get_state(review_config)
print('next:', paused_snapshot.next)
print('task interrupts:', paused_snapshot.tasks[0].interrupts)


Resume with `Command(resume=...)` and the **same** thread ID. The resume value becomes the return value of `interrupt(...)` inside `human_review`. That node restarts from its first line, which is why irreversible work belongs after the interrupt or behind an idempotency key.


In [ ]:
released = refund_graph.invoke(Command(resume='approve'), review_config)
print('decision:', released['decision'])
print('audit:', released['audit_log'])
print('next:', refund_graph.get_state(review_config).next)


### Correct earlier state and fork the timeline

Suppose the amount was entered incorrectly. We locate the checkpoint immediately before the parallel checks, create a new checkpoint with `$80.00`, and continue from there.

`update_state` does **not** edit the old snapshot. It returns a configuration for a new branch. Invoking with `None` means: continue from the work already stored in this checkpoint. The checks run again; normalization and earlier work do not.


In [ ]:
review_history = list(refund_graph.get_state_history(review_config))
before_checks = next(
    snapshot
    for snapshot in review_history
    if set(snapshot.next) == {'policy_check', 'amount_check'}
)

fork_config = refund_graph.update_state(
    before_checks.config,
    {
        'amount_usd': 80.00,
        'audit_log': ['correction:80.00'],
    },
    as_node='normalize',
)

fork_snapshot = refund_graph.get_state(fork_config)
print('fork source:', fork_snapshot.metadata['source'])
print('fork amount:', fork_snapshot.values['amount_usd'])
print('fork next:', fork_snapshot.next)

forked = refund_graph.invoke(None, fork_config)
print('original decision:', released['decision'], 'amount:', released['amount_usd'])
print('fork decision:', forked['decision'], 'amount:', forked['amount_usd'])
print('fork audit:', forked['audit_log'])


### What the runtime did—and did not—do

| Concern | This lab | Production choice |
|---|---|---|
| Workflow identity | Stable `thread_id` strings | Use your application case or job ID |
| Persistence | `InMemorySaver` | Durable database checkpointer |
| Audit | Reducer-backed state plus checkpoint history | Retention, access control, and export policy |
| Side effects | None before the interrupt | Idempotency keys and transactional boundaries |
| Intelligence | Deterministic Python rules | Add a model only where judgment is actually needed |
| Model calls | 0 | Measure separately if a model node is added |
| API cost | `$0.00` | Checkpoint storage and application infrastructure still cost money |

A graph does not make policy correct. It makes execution boundaries and saved state explicit. Your application still owns identities, authorization, validation, side effects, retention, and recovery procedures.


## 4. Challenge — reject a multi-signal refund

Process `case-201`: `$800`, 45 days since purchase, item opened. Use a new thread ID. Stop at human review, inspect the saved snapshot, then resume with `reject`.

Acceptance criteria:

1. `challenge_paused` contains an interrupt.
2. `challenge_payload` contains all three signals: `high_amount`, `outside_30_day_window`, and `opened_item`.
3. `challenge_snapshot.next` is `('human_review',)`.
4. `challenge_result['decision']` is `rejected`.
5. The audit ends with `final:rejected`.

Do not modify the graph or the node functions. Use the persistence and resume interfaces you just observed.


In [ ]:
challenge_config = {'configurable': {'thread_id': 'bonus-case-201'}}
challenge_case = new_case('case-201', amount_usd=800, days=45, opened=True)

# Complete these four lines.
# challenge_paused = ...
# challenge_payload = ...
# challenge_snapshot = ...
# challenge_result = ...


In [ ]:
expected_signals = {'high_amount', 'outside_30_day_window', 'opened_item'}
assert '__interrupt__' in challenge_paused
assert challenge_payload['case_id'] == 'case-201'
assert set(challenge_payload['signals']) == expected_signals
assert challenge_snapshot.next == ('human_review',)
assert challenge_result['decision'] == 'rejected'
assert challenge_result['audit_log'][-1] == 'final:rejected'
print('challenge passed')


## Takeaway

Use LangGraph when explicit state transitions, durable pauses, inspection, replay, or custom topology are part of the product requirement. Do not add a graph merely because the application calls a model. A fixed pipeline remains a pipeline, and a standard agent loop may be clearer through a higher-level SDK.

Current API references: [persistence](https://docs.langchain.com/oss/python/langgraph/persistence), [interrupts](https://docs.langchain.com/oss/python/langgraph/interrupts), and [streaming](https://docs.langchain.com/oss/python/langgraph/streaming).
